# 검색 베이스라인 돌려보기

질문 하나가 ① 쪼개기 → ② 분류 → ③ 라우터 → ④ 검색 → `LlmRequest` 까지 가는 과정을 단계별로 봐요.

먼저 필요한 것: DB(`docker compose up -d`), `../goldenset` 저장소, `pip install -r requirements.txt`

In [28]:
!pip install matplotlib-venn

In [29]:
!git clone -q https://github.com/Jaeyong-s-HR-Chatbot/goldenset /content/goldenset
import os
os.environ["GOLDENSET_DIR"] = "/content/goldenset"

fatal: destination path '/content/goldenset' already exists and is not an empty directory.


In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
%cd /content/drive/MyDrive/Team\ 1\ Project\ 3
!pip install -q -r requirements.txt

/content/drive/MyDrive/Team 1 Project 3


In [32]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # 저장소 루트가 현재 폴더인 경우

try:
    from rag import Pipeline, load_config, split_question

    cfg = load_config()
    pipe = Pipeline(cfg)      # BGE-M3 불러오기 + BM25 색인 (처음엔 1~2분)
    len(pipe.retriever.rows)  # 검색 대상 조각 수
except ModuleNotFoundError as e:
    print("\n[오류 발생] 'rag' 모듈을 찾을 수 없습니다.")
    print("원인: 현재 작업 디렉토리(/content)에 소스 코드가 클론(clone)되거나 업로드되지 않았습니다.")
    print("해결 방법: 먼저 아래에 준비된 셀에서 저장소를 클론한 뒤에 이 셀을 다시 실행해 주세요.\n")
    raise e

ImportError: cannot import name 'Pipeline' from 'rag' (unknown location)

In [ ]:
import os, sys
print(os.getcwd())
print(os.listdir('.'))
print(os.listdir('rag') if os.path.isdir('rag') else "rag 폴더 없음")
!find . -maxdepth 3 -name "__init__.py" -path "*rag*"
print([p for p in sys.path if os.path.isdir(os.path.join(p, 'rag'))])

In [ ]:
import os
print("현재 작업 디렉토리:", os.getcwd())
print("현재 디렉토리 내 파일 및 폴더 목록:", os.listdir('.'))

In [ ]:
# 저장소 주소가 있다면 아래 주소를 수정하여 실행하세요.
# !git clone <Jaeyong-s-HR-Chatbot-rag> .
# 예: !git clone https://github.com/Jaeyong-s-HR-Chatbot/Jaeyong-s-HR-Chatbot-rag .
# 만약 특정 폴더 안에 들어왔다면 %cd <폴더명> 명령어로 이동해야 할 수 있습니다.

## ① 질문 쪼개기

In [ ]:
try:
    for q in ["결혼하면 휴가 며칠이고 경조금 얼마야?",
              "5일 썼는데, 며칠까지 돼요?",
              "복지포인트는 언제 들어오고 어디서 쓸 수 있어요?"]:
        print(q, "→", split_question(q))
except NameError as e:
    print("[오류] 'split_question' 함수가 정의되지 않았습니다.")
    print("원인: 이전 셀에서 'rag' 모듈 임포트(ImportError)가 실패하여 함수를 가져오지 못했습니다.")
    print("해결: 위쪽의 'from rag import ...' 임포트 셀을 먼저 성공적으로 실행해 주세요.")

## ②③ 분류기와 라우터

In [ ]:
q = "결혼하면 휴가 며칠이고 경조금 얼마야?"
r = pipe.run(q)
print("상태:", r.status, r.message or "")
for s in r.subs:
    print(s.question)
    print("  카테고리:", [(c, round(x, 3)) for c, x in s.query_class.categories], "confident =", s.query_class.confident)
    print("  전체 점수:", {c: round(x, 3) for c, x in sorted(s.query_class.scores.items(), key=lambda kv: -kv[1])})
    print("  slots:", s.query_class.slots, "| 라우터:", s.route)

In [ ]:
import sys
from pathlib import Path

# 원래 경로가 아닌 실제 패키지 루트(하위 rag 폴더)를 경로에 추가합니다.
sys.path.insert(0, str(Path.cwd() / "rag"))

try:
    from rag import Pipeline, load_config, split_question

    cfg = load_config()
    pipe = Pipeline(cfg)      # BGE-M3 불러오기 + BM25 색인 (처음엔 1~2분)
    print("검색 대상 조각 수:", len(pipe.retriever.rows))
except ModuleNotFoundError as e:
    print("\n[오류 발생] 'rag' 모듈을 찾을 수 없습니다.")
    raise e


## ④ 검색 후보 (최대 20개)

In [ ]:
import pandas as pd

s = r.subs[0]
print("필터:", s.search.used_filter, s.search.filter_categories, "| fallback:", s.search.used_fallback,
      "| 1등 코사인:", s.search.top_cos, "|", s.search.doc_kind_counts)
df = pd.DataFrame([c.to_dict() for c in s.search.candidates])
df.insert(1, "title", [pipe.retriever.rows[c]["doc_name"] + " " + (pipe.retriever.rows[c]["article"] or "") for c in df.chunk_id])
df

## LLM 에 넘길 것: ChunkHit 8개 → LlmRequest

In [ ]:
from rag.goldenset import build_messages

for c in r.chunks:
    print(c.rank, c.chunk_id, c.title, round(c.score, 3), c.categories)
print()
print(build_messages(r.llm_request)[1]["content"][:800])

## 실험 구성 바꿔 보기 (A0 / A1 / B / C)

모델과 색인은 그대로 두고 스위치만 바꿔요.

In [ ]:
q = "서울 출장 가면 숙박비 얼마까지 나와?"
for name in ["A0", "A1", "B", "C"]:
    rr = pipe.with_config(cfg.preset(name)).run(q)
    print(name, [c.chunk_id for c in rr.chunks[:5]])

## 범위 밖 · 애매함 · 팀 단위

In [ ]:
for q in ["요즘 주식 뭐 사야 돼?", "휴가 며칠이야?", "우리 팀 연차 현황 보여줘", "나 올해 연차 몇 개야?"]:
    rr = pipe.run(q)
    print(f"{q} → {rr.status} | hr_needed={rr.hr_needed} | {rr.message or ''}")